# 连续吸引子网络与慢适应

下面运行 A-CANN 的三组示例：平滑跟踪、振荡跟踪和行波。环境安装见 [README](README.md)。

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from biai.cann_sfa.cann_sfa_triple_regimes import (
    REGIMES,
    animate_regime,
    plot_regime,
    simulate_regime,
)

## 1. 模型

第 $i$ 个神经元的偏好位置为 $x_i=-\pi+2\pi i/N$。用 $u_i\geq0$ 表示它的活动，$v_i$ 表示适应引起的负反馈。神经元的输出 $r_i$ 随 $u_i^2$ 增大，同时受到整个网络活动的抑制：

$$r_i=\frac{g u_i^2}{1+k\sum_j u_j^2}.$$

其中 $g$ 是输出增益，$k$ 控制全局抑制的强度。活动和适应按下面的方程变化：

$$\tau\frac{du_i}{dt}=-u_i+\sum_j J_{ij}r_j+I_i-v_i,\qquad
\tau_v\frac{dv_i}{dt}=-v_i+m u_i.$$

$u_i$ 接收其他神经元的输出和外部输入 $I_i$，受到 $v_i$ 的抑制。$v_i$ 随 $u_i$ 增大，反馈强度由 $m$ 控制。我们取 $\tau_v>\tau$，使适应滞后于活动。

偏好位置越接近，两个神经元之间的兴奋性连接越强。外部输入也是一个局部峰，中心位于目标位置 $z_{\rm ext}(t)$：

$$J_{ij}=\frac{1}{\sqrt{2\pi}a}\exp\!\left[-\frac{d(x_i-x_j)^2}{2a^2}\right],\qquad
I_i(t)=\alpha\exp\!\left[-\frac{d(x_i-z_{\rm ext}(t))^2}{4a^2}\right].$$

这里 $d(\theta)=((\theta+\pi)\bmod 2\pi)-\pi$ 给出圆环上的有符号最短距离，$a$ 控制宽度，$\alpha$ 是输入幅度。目标以恒定速度移动：$z_{\rm ext}(t)=z_0+v_{\rm ext}t$。时间单位为 ms，位置为 rad，速度为 rad/ms。

以下实验固定使用 $N=256$ 个神经元。

## 2. 实验参数

每次实验都从 $u=v=0$ 开始。前两组始终有移动目标；第三组的目标只出现 30 ms，随后输入归零。每组运行 1400 ms，步长为 0.1 ms。

| 实验 | 反馈强度 $m$ | 输入幅度 $\alpha$ | 目标速度（rad/ms） | 输入持续时间 |
| --- | ---: | ---: | ---: | ---: |
| 平滑跟踪 | 0.05 | 0.20 | 0.004 | 全程 |
| 振荡跟踪 | 16.7/50 | 0.20 | 0.0012 | 全程 |
| 行波 | 0.90 | 0.03 | 0.004 | 30 ms |

这些参数分别产生三种状态。第 6 节会固定输入和速度，单独考察反馈强度的作用。

In [ ]:
configs = {
    "smooth": replace(REGIMES["smooth"], m=0.05, alpha=0.2, v_ext=0.004),
    "oscillatory": replace(REGIMES["oscillatory"], m=16.7 / 50.0, alpha=0.2, v_ext=0.0012),
    "travelling_wave": replace(
        REGIMES["travelling_wave"], m=0.9, alpha=0.03, v_ext=0.004, stim_duration=30.0
    ),
}
results = {name: simulate_regime(config) for name, config in configs.items()}

## 3. 平滑跟踪

运行下面的单元格，可以看到活动峰逐渐跟上目标，随后与目标保持近似固定的距离。

第一幅图画出活动峰和目标的位置。位置经过 $\pi$ 后回到 $-\pi$，曲线在此处断开。第二幅图画跟踪误差，正值表示领先，负值表示落后。计算误差时保留完整的绕圈轨迹，因此经过圆环边界不会产生跳变。第三幅图是活动热图，颜色对应色条上的活动值 $u$。

活动峰的中心用圆周平均计算：$z=\arg\sum_i u_i e^{\mathrm{i}x_i}$。没有活动，或圆周向量接近零时，中心记为 NaN。

静态图下方的动画同时显示外部输入和活动峰的形状。点击播放键即可观看，也可以拖动进度条。

In [ ]:
smooth_figure = plot_regime(results["smooth"])

In [ ]:
smooth_animation = animate_regime(results["smooth"], frame_stride=120, fps=12)
display(HTML(smooth_animation.to_jshtml()))

## 4. 振荡跟踪

这一组的跟踪误差会持续振荡，活动峰时而领先目标，时而落后。活动峰经过某个位置后，该处的慢反馈仍然较强，推动活动峰离开；外部输入则把它拉回目标附近。两者共同作用，使活动峰在移动目标附近来回摆动。

热图中的亮带也随之弯曲。动画中可以看到，活动峰与输入峰交替出现在彼此的前方。

In [ ]:
oscillatory_figure = plot_regime(results["oscillatory"])

In [ ]:
oscillatory_animation = animate_regime(results["oscillatory"], frame_stride=120, fps=12)
display(HTML(oscillatory_animation.to_jshtml()))

## 5. 行波

目标向右移动 30 ms 后消失，活动峰仍沿圆环向右传播。中间一幅图保留累计位置，因而能看出活动峰走过了多少圈；虚线标出撤去输入的时刻。

这时已经没有目标，跟踪误差不再定义。活动峰的传播方向由最初的移动线索选定，将线索速度改为负值可以得到向左传播的行波。

下方还会列出各组在后半程的平均速度和跟踪误差极差。极差是最大误差与最小误差之差，可以用来比较摆动幅度。

In [ ]:
wave_figure = plot_regime(results["travelling_wave"])

# Measure the second half of each trial, after the initial transient.
for name, result in results.items():
    late = result["ts"] > result["config"].total_duration / 2
    defined = late & np.isfinite(result["center"])
    speed = np.polyfit(result["ts"][defined], result["center"][defined], 1)[0]
    errors = result["relative"][late & np.isfinite(result["relative"])]
    span = f"{np.ptp(errors):.4f} rad" if errors.size else "无目标"
    print(f"{name}: 平均速度 {speed:.5f} rad/ms；误差极差 {span}")

In [ ]:
travelling_wave_animation = animate_regime(results["travelling_wave"], frame_stride=120, fps=12)
display(HTML(travelling_wave_animation.to_jshtml()))

## 6. 去掉慢反馈

把振荡跟踪组的 $m$ 改为零，其余参数不变。去掉反馈后，误差逐渐趋于稳定；有反馈时，误差持续振荡。两条曲线共用输入和目标速度，便于比较慢反馈的作用。

In [ ]:
no_feedback = simulate_regime(replace(configs["oscillatory"], name="No adaptation", m=0.0))
feedback_figure, ax = plt.subplots(figsize=(10, 3))
for result, label in ((no_feedback, "m = 0"), (results["oscillatory"], "m = 16.7 / 50")):
    ax.plot(result["ts"], result["relative"], label=label)
ax.set(xlabel="time (ms)", ylabel="bump - applied target (rad)", title="Effect of slow feedback")
ax.legend()
feedback_figure.tight_layout()
plt.show()

## 附录：数值计算

每个时间步先用旧状态计算 $r_n$ 和网络内部的输入 $Jr_n$，再用指数 Euler 法更新：

$$u_{n+1}=\max\{u_n+(1-e^{-\Delta t/\tau})(Jr_n+I_n-v_n-u_n),0\},$$
$$v_{n+1}=v_n+(1-e^{-\Delta t/\tau_v})(m u_n-v_n).$$

两个式子都使用更新前的 $u_n,v_n$。这种算法对泄漏项精确积分，并把其余驱动力在一步内视为常数。例如 $\tau=1$、$\Delta t=0.1$ 时，活动更新系数为 $1-e^{-0.1}\approx0.09516$。更新后的活动若小于零，就截为零。

代码中的 `input_ts[n]` 是输入开始的时间，`ts[n]=input_ts[n]+dt` 是更新后状态的时间。跟踪误差比较这个状态与刚施加的目标位置。撤去输入后，目标位置和误差都记为 NaN。

### 用周期卷积计算连接输入

神经元在圆环上均匀排列，连接只取决于相对位置，因此连接矩阵的每一行都是上一行的循环平移。令第一列为 $K$，就可以通过快速傅里叶变换（FFT）计算连接输入：

$$Jr=\operatorname{irFFT}(\operatorname{rFFT}(K)\operatorname{rFFT}(r)).$$

这与离散矩阵乘法计算同一个和，不额外乘网格间距。每步计算量为 $O(N\log N)$，只需存储长度为 $O(N)$ 的连接核。小网络的实际耗时还取决于 FFT 和矩阵库的实现。

[仿真代码](cann_sfa_triple_regimes.py) 中，`CANN1DSFA.update` 完成一步更新，`build_inputs` 生成输入，`simulate_regime` 串起整个实验。

## 阅读材料

Chu 等人的 [NeurIPS 2022 论文](https://papers.nips.cc/paper_files/paper/2022/hash/d6797a91df19b768409b5178642dcb26-Abstract-Conference.html) 讨论了带反馈抑制的连续吸引子网络如何产生振荡跟踪，并用它解释海马位置细胞的相位变化。本课选取三组参数演示网络动力学；论文中的生物解释和完整相图可继续阅读原文。

## 可选拓展

- 固定其他参数，逐步改变 $m$，记录后半程跟踪误差的极差，找出持续振荡出现的范围。
- 将步长减半，比较活动峰速度和振荡幅度的变化。
- 将行波的初始线索速度改为负值，观察撤去输入后的传播方向。